# PDF Page Rotator
원하는 페이지를 선택하여 회전시키고 저장합니다.

In [ ]:
# Install if needed
# !pip install pypdf tqdm

In [ ]:
from pypdf import PdfReader, PdfWriter
from tqdm import tqdm
import os

In [ ]:
# ── Settings ──────────────────────────────────────────────
PDF_PATH   = "input.pdf"   # PDF 경로
ANGLE      = 90            # 회전 각도: 90, 180, 270 (시계방향)

# 회전할 페이지 지정 (1-indexed)
# 예) [1, 3, 5]  또는  "all"  또는  "1-5"  또는  "1,3,5-8"
PAGES      = "all"

# 출력 경로 (None이면 자동 생성)
OUTPUT_PATH = None
# ──────────────────────────────────────────────────────────

In [ ]:
def parse_pages(spec, total):
    """Parse page spec to 0-indexed set.
    Accepts: 'all', list of ints, or string like '1,3,5-8'
    """
    if spec == "all":
        return set(range(total))
    if isinstance(spec, list):
        return {p - 1 for p in spec}
    # string parsing
    result = set()
    for part in str(spec).split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-")
            result.update(range(int(a) - 1, int(b)))
        else:
            result.add(int(part) - 1)
    return result


def rotate_pdf(pdf_path, pages_spec, angle, output_path=None):
    assert angle in (90, 180, 270), "angle must be 90, 180, or 270"
    assert os.path.exists(pdf_path), f"File not found: {pdf_path}"

    reader = PdfReader(pdf_path)
    total  = len(reader.pages)
    target = parse_pages(pages_spec, total)

    # validate
    out_of_range = [p + 1 for p in target if p < 0 or p >= total]
    if out_of_range:
        raise ValueError(f"Out-of-range pages (total={total}): {out_of_range}")

    writer = PdfWriter()
    for i, page in enumerate(tqdm(reader.pages, desc="Processing pages")):
        if i in target:
            page.rotate(angle)
        writer.add_page(page)

    if output_path is None:
        base, ext = os.path.splitext(pdf_path)
        output_path = f"{base}_rotated{ext}"

    with open(output_path, "wb") as f:
        writer.write(f)

    rotated = sorted(p + 1 for p in target)
    print(f"\nDone! Rotated {len(target)} page(s) by {angle}°: {rotated}")
    print(f"Saved → {output_path}")
    return output_path

In [ ]:
# Preview: show total page count
reader = PdfReader(PDF_PATH)
print(f"Total pages: {len(reader.pages)}")

In [ ]:
rotate_pdf(PDF_PATH, PAGES, ANGLE, OUTPUT_PATH)